In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 200)

In [2]:
train = pd.read_parquet('../dataset/train.parquet')
benchmark_items = pd.read_parquet('../dataset/benchmark_items.parquet')
benchmark_queries = pd.read_parquet('../dataset/benchmark_queries.parquet')

print("train:", train.shape)
print("benchmark_items:", benchmark_items.shape)
print("benchmark_queries:", benchmark_queries.shape)

train: (497673, 19)
benchmark_items: (189212, 14)
benchmark_queries: (2452, 6)


In [3]:
def basic_info(df, name):
    print("=" * 80)
    print(f"{name}: shape = {df.shape}")
    print("\nDtypes:")
    print(df.dtypes)
    print("\nПропуски:")
    print(df.isna().sum())
    print("\nNunique:")
    print(df.nunique())
    print()

basic_info(train, "train")
basic_info(benchmark_items, "benchmark_items")
basic_info(benchmark_queries, "benchmark_queries")

train: shape = (497673, 19)

Dtypes:
search_query                     str
search_location_id             int64
search_is_delivery_search      int32
search_infm_params_text          str
search_category                int64
item_title_raw                   str
item_rating_reviews_count    float64
item_rating                  float64
item_price                    object
item_microcat_id               int64
item_longitude                object
item_location_id               int64
item_latitude                 object
item_is_phone_hidden            bool
item_is_message_forbidden       bool
item_infm_params_text            str
item_id                          str
item_description_raw             str
item_category_id               int64
dtype: object

Пропуски:
search_query                     0
search_location_id               0
search_is_delivery_search        0
search_infm_params_text          0
search_category                  0
item_title_raw                   0
item_rating_reviews_count

In [4]:
print("=== train head ===")
display(train.head(5))
print("\n=== benchmark_items head ===")
display(benchmark_items.head(5))
print("\n=== benchmark_queries head ===")
display(benchmark_queries.head(5))

=== train head ===


,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,скупка телевизоров,652430,0,,114,Скупка б/у техники,91.0,4.989011,1.000000000000000,2303374,39.712818145751953,652430,54.629230499267578,True,False,"Вид услуги Место оказания услуг Первомайский пр-т, 66 Тип стоимости за услугу Работаете с юрлицами и ИП Опыт работы 10 лет и больше Гарантия на выполнение работ Как вы работаете У себя Предоплата ...",e8b685dffe1a408e,"Скупаю практически любую современную новую и б/у технику(обязательно рабочую), до 80% от рыночной стоимости.\nВсе предложения присылайте в личные сообщения Авито, с указанием точной модели и фото....",114
1,автоподбор,640860,0,Рейтинг пользователя 4 звезды и выше,114,Автоподбор Разовый осмотр автомобиля,462.0,4.982684,3500.000000000000000,2303374,44.051009999999998,640860,56.273389999999999,False,False,"Вид услуги Место оказания услуг Нижний Новгород, Советский район, жилой комплекс Новая Кузнечиха Тип стоимости за услугу Начальная цена Работаете с юрлицами и ИП Опыт работы 10 лет и больше Берёте...",92e1b0446f827b59,🚗 Автоподбор и выездная диагностика автомобиля в Нижнем Новгороде и области.\n\n🫴 Помогу подобрать оптимальный вариант автомобиля под ваш бюджет.\n\n🔦 Произведу разовый осмотр автомобиля перед пок...,114
2,баня на дровах,653240,0,"Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье",114,"Баня на дровах ""Прованс"" на Цветочной",NaN,NaN,1600.000000000000000,86469,30.128784180000000,653240,59.783687590000000,False,False,"Вид услуги Красота, здоровье Место оказания услуг Санкт-Петербург, садоводческое некоммерческое товарищество Веретено-2, Цветочная ул., 146 Тип услуги СПА-услуги, массаж Тип стоимости за час Где в...",624846856ce81d69,"В ритме современной жизни так сложно найти момент, чтобы остановиться: выключить телефон, отложить дела и подарить себе и своим близким настоящее, ни с чем не сравнимое удовольствие — погрузиться...",114
3,изготовление госномера на авто,634670,0,"Вид услуги Оборудование, производство",114,"Изготовление дубликатов авто номеров, гос номеров",14.0,4.714286,1700.000000000000000,2303428,40.537841000000000,633570,45.424875000000000,False,False,"Вид услуги Оборудование, производство Тип услуги Производство, обработка Место оказания услуг Краснодарский край, Кавказский район, Кропоткин Тип стоимости за услугу Начальная цена График работы о...",45b8628b9c6e7b85,"Изготовим дубликат номера по утере или износу на официальных основаниях. \r\nЛЮБОЙ РЕГИОН России 🇷🇺.\r\nИзготовление по ГОСТ в соответствии с требованиями МВД. Все степени защиты, любой регион.\n\...",114
4,укладка плитки,658430,0,Тип услуги Ремонт квартир и домов под ключ Вид услуги Ремонт и отделка,114,Ремонт и отделка квартир под ключ,1.0,5.000000,1000.000000000000000,44725,69.496444699999998,658430,56.105983729999998,False,False,Вид услуги Ремонт и отделка Тип услуги Ремонт квартир и домов под ключ Место оказания услуг ул. Полины Осипенко Тип стоимости за услугу Работа по договору Гарантия Проживание на объекте Нет Бригад...,c95a4a7daf2a967f,отделочные работы любой сложности.,114



=== benchmark_items head ===


,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id
0,Ремонт/выкуп компьют. и ноутбуков с выездом на дом,57.0,5.0,500.000000000000000,2097573,42.047193961004901,631060,44.228180450924000,False,False,"Вид услуги Компьютерная помощь Место оказания услуг пр-т Ленина Тип стоимости за услугу Начальная цена График работы от 34200 График работы до 82800 Время работы, с 09:30 Время работы, до 23:00 На...",111eb8b979577d79,"Выезд на дом в любое время, вплоть до 23:00\n\n● Диагностика устройств;\n\n● Чистка от пыли и замена термопасты;\n\n● Проверка работоспособности всех компонентов.\n\nРемонт аппаратной части:\n\n● ...",114
1,Обучение ребенка чтению,7.0,5.0,900.000000000000000,86453,49.627391820000000,631870,58.627609249999999,False,False,"Вид услуги Обучение, курсы Место оказания услуг ул. Чернышевского, 35 Тип услуги Детское развитие, логопеды Тип стоимости за урок Начальная цена График работы от 27000 График работы до 70200 Время...",75fc8e10f5a66fc4,"Дорогие родители и маленькие книголюбы! \n\nЯ, детский психолог/нейропсихолог, рада пригласить вас и ваших детей на захватывающее и увлекательное обучение чтению в рамках программы ""Словолодочки"" ...",114
2,Афрокудри 5+,14.0,5.0,1000.000000000000000,86467,40.936892000000000,628500,56.990307000000001,True,False,"Вид услуги Красота, здоровье Место оказания услуг городской округ Иваново, Фрунзенский район Тип услуги Услуги парикмахера Тип стоимости за услугу Начальная цена Где вы оказываете услуги У заказчи...",3f6ae81704565b9c,"ВНИМАНИЕ Укладка !!!! 🥳\n\nАФРОкудри отличный вариант замены хим завивки, если хочется поменять временно свой образ.\n\nДля взрослых и детей! \n\nДержится укладка до мытья головы, время работы от ...",114
3,Монтаж малых архитектурных форм,2.0,5.0,5000.000000000000000,2058739,37.576183000000000,637640,55.662734999999998,False,False,"Вид услуги Строительство Место оказания услуг Севастопольский пр-т, 28к4 Тип стоимости за услугу Работа по договору Гарантия График работы от 28800 График работы до 72000 График работы, дни недели...",0618dec37a59a21a,"Оказываем все виды услуг по сборке и монтажу игровых площадок , а так же все виды работ по устройству основания и ограждения.",114
4,Репетитор по истории 10 класс ЕГЭ,19.0,5.0,1500.000000000000000,86456,44.750473999999997,624850,48.786008000000002,False,False,"Вид услуги Обучение, курсы Место оказания услуг Волгоградская обл., Волжский, пл. имени В. И. Ленина Тип услуги Предметы школы и вуза Начальная цена График работы от 0 График работы до 86340 Время...",13da2c81574677ed,🚀 Сдай ЕГЭ по истории на 80+ за 6 месяцев!\n\nМечтаешь поступить в вуз своей мечты? ✨\n\nИстория больше не будет скучной и сложной — я помогу тебе превратить подготовку в увлекательное путешествие...,114



=== benchmark_queries head ===


,query_id,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category
0,70DfDUpwjxB4lzFd,перевозки владикавказ тбилиси,649820,0,,114
1,JTrdTaZJvSiLPkXj,обзвон по базе,107620,0,Вид услуги Деловые услуги,114
2,LZCZNoVG4AFUkVRJ,липоредукция подбородка,637640,0,"Вид услуги Красота, здоровье",114
3,660ac9QVtXkRxZC3,подъемник 4 х стоечный,662810,0,,114
4,YgHcM9MVbxKnxD1e,монтаж видеодомофонов,642790,0,,114


In [5]:
print("search_category в train:")
print(train['search_category'].value_counts(dropna=False))

print("\nsearch_category в benchmark_queries:")
print(benchmark_queries['search_category'].value_counts(dropna=False))

print("\nsearch_is_delivery_search в train:")
print(train['search_is_delivery_search'].value_counts(dropna=False))

print("\nsearch_is_delivery_search в benchmark_queries:")
print(benchmark_queries['search_is_delivery_search'].value_counts(dropna=False))

print("\nitem_category_id в benchmark_items (топ-20):")
print(benchmark_items['item_category_id'].value_counts(dropna=False).head(20))

print("\nitem_microcat_id в benchmark_items (топ-20):")
print(benchmark_items['item_microcat_id'].value_counts(dropna=False).head(20))

search_category в train:
search_category
114    497635
0          34
81          2
10          2
Name: count, dtype: int64

search_category в benchmark_queries:
search_category
114    2230
0       222
Name: count, dtype: int64

search_is_delivery_search в train:
search_is_delivery_search
0    497661
1        12
Name: count, dtype: int64

search_is_delivery_search в benchmark_queries:
search_is_delivery_search
0    2452
Name: count, dtype: int64

item_category_id в benchmark_items (топ-20):
item_category_id
114    187336
19        357
112       141
40        122
33        103
10         98
20         84
25         82
36         74
111        74
39         72
24         70
83         68
27         57
30         56
97         44
21         36
88         34
82         32
42         29
Name: count, dtype: int64

item_microcat_id в benchmark_items (топ-20):
item_microcat_id
86456      6355
2301563    4272
86467      4064
86470      3858
1289835    3696
4141       3417
2303435    3267
86458  

In [6]:
def check_ids(df, col, name):
    s = df[col].astype(str)
    print(f"\n{name}.{col}:")
    print("Длины:", s.str.len().value_counts().head().to_dict())
    print("Все в нижнем регистре?", bool((s == s.str.lower()).all()))
    print("Все hex?", bool(s.str.match(r'^[0-9a-f]+$').all()))
    print("Уникальных:", s.nunique(), "из", len(s))

check_ids(train, 'item_id', 'train')
check_ids(benchmark_items, 'item_id', 'benchmark_items')
check_ids(benchmark_queries, 'query_id', 'benchmark_queries')


train.item_id:
Длины: {16: 497673}
Все в нижнем регистре? True
Все hex? True
Уникальных: 344825 из 497673

benchmark_items.item_id:
Длины: {16: 189212}
Все в нижнем регистре? True
Все hex? True
Уникальных: 189212 из 189212

benchmark_queries.query_id:
Длины: {16: 2452}
Все в нижнем регистре? False
Все hex? False
Уникальных: 2452 из 2452


In [7]:
train_items = set(train['item_id'].astype(str))
bench_items = set(benchmark_items['item_id'].astype(str))

print("Пересечение item_id train и benchmark_items:", len(train_items & bench_items))
print("item_id из train, которых нет в benchmark_items:", len(train_items - bench_items))
print("item_id из benchmark_items, которых нет в train:", len(bench_items - train_items))

train_q = set(train['search_query'].astype(str))
bench_q = set(benchmark_queries['search_query'].astype(str))

print("\nУникальных search_query в train:", len(train_q))
print("Уникальных search_query в benchmark_queries:", len(bench_q))
print("Пересечение search_query:", len(train_q & bench_q))
print("Доля benchmark-запросов, чей текст есть в train:",
      round(benchmark_queries['search_query'].astype(str).isin(train_q).mean(), 4))

Пересечение item_id train и benchmark_items: 18142
item_id из train, которых нет в benchmark_items: 326683
item_id из benchmark_items, которых нет в train: 171070

Уникальных search_query в train: 74529
Уникальных search_query в benchmark_queries: 2452
Пересечение search_query: 907
Доля benchmark-запросов, чей текст есть в train: 0.3699


In [8]:
def make_query_key(df):
    cols = ['search_query', 'search_location_id', 'search_is_delivery_search',
            'search_infm_params_text', 'search_category']
    return df[cols].fillna('__NA__').astype(str).agg('||'.join, axis=1)

train['query_key'] = make_query_key(train)
benchmark_queries['query_key'] = make_query_key(benchmark_queries)

print("train: уникальных query_key =", train['query_key'].nunique())
print("benchmark_queries: уникальных query_key =", benchmark_queries['query_key'].nunique())
print("benchmark_queries: уникальных query_id =", benchmark_queries['query_id'].nunique())

train: уникальных query_key = 354463
benchmark_queries: уникальных query_key = 2452
benchmark_queries: уникальных query_id = 2452


In [9]:
g = train.groupby('query_key').agg(
    n_rows=('item_id', 'size'),
    n_items=('item_id', 'nunique'),
    search_query=('search_query', 'first'),
    location=('search_location_id', 'first'),
    category=('search_category', 'first'),
).reset_index()

print("Распределение числа позитивов на query_key:")
print(g['n_items'].describe())

print("\nТоп значений n_items:")
print(g['n_items'].value_counts().sort_index().head(20))

print("\nДоля запросов с >1 позитивом:", round((g['n_items'] > 1).mean(), 4))
print("Доля запросов с >2 позитивами:", round((g['n_items'] > 2).mean(), 4))
print("Максимум позитивов на запрос:", g['n_items'].max())

Распределение числа позитивов на query_key:
count    354463.000000
mean          1.317638
std           1.611180
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max         278.000000
Name: n_items, dtype: float64

Топ значений n_items:
n_items
1     301297
2      33624
3       9343
4       3922
5       2019
6       1188
7        737
8        543
9        388
10       255
11       186
12       168
13       117
14        94
15        80
16        69
17        51
18        39
19        39
20        34
Name: count, dtype: int64

Доля запросов с >1 позитивом: 0.15
Доля запросов с >2 позитивами: 0.0551
Максимум позитивов на запрос: 278


In [10]:
dup_pairs = train.duplicated(subset=['query_key', 'item_id']).sum()
print("Дубли пар query_key + item_id в train:", dup_pairs)

Дубли пар query_key + item_id в train: 30619


In [11]:
train['cat_match'] = train['search_category'].astype(str) == train['item_category_id'].astype(str)
train['loc_match'] = train['search_location_id'].astype(str) == train['item_location_id'].astype(str)

print("Доля строк, где search_category == item_category_id:", round(train['cat_match'].mean(), 4))
print("Доля строк, где search_location_id == item_location_id:", round(train['loc_match'].mean(), 4))

Доля строк, где search_category == item_category_id: 0.9999
Доля строк, где search_location_id == item_location_id: 0.831


In [12]:
q_stats = train.groupby('search_query').agg(
    n_rows=('item_id', 'size'),
    n_unique_keys=('query_key', 'nunique'),
    n_unique_items=('item_id', 'nunique'),
    n_locations=('search_location_id', 'nunique'),
    n_infm=('search_infm_params_text', 'nunique'),
).reset_index()

print("Статистика по search_query:")
print(q_stats.describe())

print("\nТоп-20 search_query по числу строк:")
print(q_stats.sort_values('n_rows', ascending=False).head(20).to_string(index=False))

Статистика по search_query:
             n_rows  n_unique_keys  n_unique_items   n_locations        n_infm
count  74529.000000   74529.000000    74529.000000  74529.000000  74529.000000
mean       6.677575       4.756041        5.974332      4.126608      1.448523
std       64.865220      25.811141       50.302240     17.022624      1.630055
min        1.000000       1.000000        1.000000      1.000000      1.000000
25%        1.000000       1.000000        1.000000      1.000000      1.000000
50%        1.000000       1.000000        1.000000      1.000000      1.000000
75%        3.000000       2.000000        3.000000      2.000000      1.000000
max     6540.000000    1693.000000     5474.000000    760.000000    166.000000

Топ-20 search_query по числу строк:
           search_query  n_rows  n_unique_keys  n_unique_items  n_locations  n_infm
                маникюр    6540           1499            5474          701     142
                 массаж    5702           1693          

In [13]:
print("Топ-20 search_infm_params_text в train:")
print(train['search_infm_params_text'].fillna('__EMPTY__').value_counts().head(20))

print("\nТоп-20 search_infm_params_text в benchmark_queries:")
print(benchmark_queries['search_infm_params_text'].fillna('__EMPTY__').value_counts().head(20))

Топ-20 search_infm_params_text в train:
search_infm_params_text
                                                                                                        163999
Вид услуги Автосервис, аренда Тип услуги Автосервис                                                      19654
Вид услуги Ремонт и отделка                                                                              15586
Вид услуги Красота, здоровье                                                                             15355
Вид услуги Праздники, мероприятия                                                                        12474
Тип услуги автосервиса Диагностика и ремонт авто Вид услуги Автосервис, аренда Тип услуги Автосервис     11644
Тип услуги Маникюр, педикюр Вид услуги Красота, здоровье                                                 11589
Вид услуги Обучение, курсы                                                                               11080
Вид услуги                                      

In [14]:
print("Топ-10 item_infm_params_text в benchmark_items (укорочено по 200 символов):")
tmp = benchmark_items['item_infm_params_text'].fillna('__EMPTY__')
for val, cnt in tmp.value_counts().head(10).items():
    print(f"\n[{cnt} раз] {val[:200]}...")

Топ-10 item_infm_params_text в benchmark_items (укорочено по 200 символов):

[36 раз] Вид услуги Вывоз мусора и вторсырья Место оказания услуг Москва График работы от 28800 График работы до 72000 График работы, дни недели пн. График работы, дни недели вт. График работы, дни недели ср. ...

[23 раз] Вид услуги Праздники, мероприятия Место оказания услуг Москва Чем вы занимаетесь Оформление мероприятий, декор...

[21 раз] Вид услуги Вывоз мусора и вторсырья Место оказания услуг Москва График работы от 28800 График работы до 79200 График работы, дни недели пн. График работы, дни недели вт. График работы, дни недели ср. ...

[18 раз] Вид услуги Вывоз мусора и вторсырья Место оказания услуг Москва График работы от 28800 График работы до 72000 График работы, дни недели пн. График работы, дни недели вт. График работы, дни недели ср. ...

[18 раз] Вид услуги Искусство Место оказания услуг Москва, проезд Воскресенские Ворота Тип стоимости за услугу Начальная цена...

[17 раз] Вид услуги Оборудо